In [1]:
# scripts/01b_boalf_reconciliation_spike.py
import requests
import polars as pl
import os
from datetime import date

# ==============================================================================
# MILESTONE 2.1b: BOALF RECONCILIATION SPIKE
# Goal: Verify BOALF dataset has boundary-level attribution for SCOTEX/SSEN-S
# ==============================================================================

# Configuration
# Resource ID for BOALF dataset (you'll need to find the correct ID from NESO CKAN)
# This is a placeholder - we need to search for the actual BOALF resource
BOALF_RESOURCE_ID = "placeholder_boalf_resource_id"  # TODO: Find actual ID
API_URL = "https://api.neso.energy/api/3/action/datastore_search"
SAMPLE_LIMIT = 1000  # Sample of individual constraint actions
TARGET_BOUNDARIES = ["SCOTEX", "SSEN-S", "B6", "B2"]
METHODOLOGY_BREAK_DATE = date(2024, 4, 22)

def search_ckan_for_boalf() -> dict:
    """
    Search CKAN for BOALF-related datasets.
    Returns package info to find the correct resource ID.
    """
    search_url = "https://api.neso.energy/api/3/action/package_search"
    params = {
        "q": "BOALF OR balancing mechanism OR constraint action",
        "rows": 10
    }
    
    print("Searching CKAN for BOALF datasets...")
    response = requests.get(search_url, params=params)
    response.raise_for_status()
    
    data = response.json()
    
    if not data.get("success"):
        raise RuntimeError(f"CKAN search failed: {data.get('error')}")
    
    results = data["result"]["results"]
    
    print(f"\nFound {len(results)} potential datasets:")
    for i, pkg in enumerate(results):
        print(f"  {i+1}. {pkg['title']}")
        print(f"     ID: {pkg['id']}")
        print(f"     Resources: {len(pkg.get('resources', []))}")
        for res in pkg.get('resources', []):
            print(f"       - {res['name']} (ID: {res['id']})")
    
    return results

def fetch_boalf_sample(resource_id: str, limit: int) -> pl.DataFrame:
    """
    Fetch a sample of BOALF data.
    """
    params = {
        "resource_id": resource_id,
        "limit": limit,
        "offset": 0
    }
    
    print(f"\nFetching BOALF sample (limit={limit}) from resource: {resource_id}")
    response = requests.get(API_URL, params=params)
    response.raise_for_status()
    
    data = response.json()
    
    if not data.get("success"):
        raise RuntimeError(f"API returned success=false: {data.get('error')}")
    
    records = data["result"]["records"]
    
    assert len(records) > 0, "No BOALF records returned. Check resource ID."
    
    return pl.DataFrame(records)

def inspect_boalf_schema(df: pl.DataFrame) -> None:
    """
    Inspect BOALF schema for boundary attribution.
    """
    print("\n" + "="*80)
    print("BOALF RECONCILIATION SPIKE RESULTS")
    print("="*80)
    
    # 1. Schema Inspection
    print("\n1. SCHEMA INSPECTION:")
    print(f"Columns available: {df.columns}")
    
    # Look for boundary/zone/location columns
    location_keywords = ["boundary", "zone", "location", "region", "area", "node", "bm"]
    location_cols = [col for col in df.columns if any(keyword in col.lower() for keyword in location_keywords)]
    print(f"Potential location/boundary columns: {location_cols}")
    
    # 2. Boundary Attribution Check
    print("\n2. BOUNDARY ATTRIBUTION CHECK:")
    if not location_cols:
        print("⚠️ CRITICAL: No boundary/zone/location column found in BOALF schema.")
        print("🚨 ACTION: This dataset may not provide geographic attribution.")
        print("🚨 PIVOT: Need to identify alternative data source or join strategy.")
    else:
        for col in location_cols:
            print(f"\n  Inspecting column: '{col}'")
            unique_vals = df[col].unique().drop_nulls().to_list()
            print(f"  Unique values (first 20): {unique_vals[:20]}")
            
            # Check for our target boundaries
            has_scotex = any("SCOTEX" in str(v).upper() or "B6" in str(v).upper() for v in unique_vals)
            has_ssen_s = any("SSEN-S" in str(v).upper() or "SSEN S" in str(v).upper() or "B2" in str(v).upper() for v in unique_vals)
            
            print(f"    -> Contains SCOTEX/B6? {has_scotex}")
            print(f"    -> Contains SSEN-S/B2? {has_ssen_s}")
            
            if has_scotex or has_ssen_s:
                print(f"    ✅ SUCCESS: This column provides boundary attribution!")
                print(f"    -> Can filter for SCOTEX and SSEN-S constraint actions.")
    
    # 3. Temporal Check
    print("\n3. TEMPORAL CHECK:")
    date_cols = [col for col in df.columns if "date" in col.lower() or "time" in col.lower()]
    if date_cols:
        print(f"Date/time columns: {date_cols}")
        # Check if data spans the methodology break
        first_date_col = date_cols[0]
        try:
            df_dates = df.with_columns(pl.col(first_date_col).str.to_date(strict=False))
            min_date = df_dates.select(pl.col(first_date_col).min()).to_series()[0]
            max_date = df_dates.select(pl.col(first_date_col).max()).to_series()[0]
            print(f"  -> Date range: {min_date} to {max_date}")
            
            if min_date <= METHODOLOGY_BREAK_DATE <= max_date:
                print(f"  -> ⚠️ WARNING: Data spans the {METHODOLOGY_BREAK_DATE} methodology break.")
        except Exception as e:
            print(f"  -> Could not parse dates: {e}")
    
    # 4. Volume/Cost Check
    print("\n4. VOLUME/COST SANITY CHECK:")
    vol_keywords = ["volume", "mwh", "mw", "quantity"]
    vol_cols = [col for col in df.columns if any(keyword in col.lower() for keyword in vol_keywords)]
    if vol_cols:
        print(f"Volume columns: {vol_cols}")
        for col in vol_cols[:2]:  # Check first 2
            try:
                df_numeric = df.with_columns(pl.col(col).cast(pl.Float64, strict=False))
                total = df_numeric.select(pl.col(col).sum()).to_series()[0]
                print(f"  -> Total '{col}' in sample: {total:,.0f}")
            except Exception as e:
                print(f"  -> Could not aggregate '{col}': {e}")
    
    print("\n" + "="*80)
    print("SPIKE COMPLETE.")
    print("="*80)

if __name__ == "__main__":
    try:
        # Step 1: Search for BOALF dataset
        search_results = search_ckan_for_boalf()
        
        # Step 2: You'll need to manually identify the correct resource ID from search results
        # and update BOALF_RESOURCE_ID above, then run:
        
        # Step 3: Fetch and inspect sample
        # sample_df = fetch_boalf_sample(BOALF_RESOURCE_ID, limit=SAMPLE_LIMIT)
        # inspect_boalf_schema(sample_df)
        
        # Step 4: Save sample for audit trail
        # os.makedirs("data/intermediate", exist_ok=True)
        # output_path = "data/intermediate/01b_boalf_sample.parquet"
        # sample_df.write_parquet(output_path)
        # print(f"\n✅ Sample data saved to: {output_path}")
        
        print("\n⚠️ ACTION REQUIRED:")
        print("1. Review the search results above")
        print("2. Identify the correct BOALF resource ID")
        print("3. Update BOALF_RESOURCE_ID in the script")
        print("4. Uncomment the fetch/inspect/save code")
        print("5. Re-run the script")
        
    except Exception as e:
        print(f"\n❌ BOALF RECONCILIATION SPIKE FAILED: {e}")
        raise

Searching CKAN for BOALF datasets...

Found 0 potential datasets:

⚠️ ACTION REQUIRED:
1. Review the search results above
2. Identify the correct BOALF resource ID
3. Update BOALF_RESOURCE_ID in the script
4. Uncomment the fetch/inspect/save code
5. Re-run the script
